### Retailpulse Silver Cleaning
#### 1. Configuration
This cell sets up the schema names for the Bronze and Silver layers in the Databricks workspace.
 - `BRONZE_SCHEMA` is used for raw, unprocessed data.
 - `SILVER_SCHEMA` is used for cleaned and transformed data.
 The configuration is printed for verification.

In [0]:
# Cell 2 - Configuration
BRONZE_SCHEMA = "workspace.retailpulse_bronze"
SILVER_SCHEMA = "workspace.retailpulse_silver"

print("Configuration loaded.")
print(f"Source: {BRONZE_SCHEMA}")
print(f"Target: {SILVER_SCHEMA}")

Configuration loaded.
Source: workspace.retailpulse_bronze
Target: workspace.retailpulse_silver


#### 2. Inspect Bronze Tables
This cell inspects the main tables in the Bronze schema:
- Iterates through `customers`, `products`, and `orders` tables.
- Prints the schema and row count for each table.
- Displays the count of null values for each column.
This helps assess data quality before transformation to the Silver layer.

In [0]:
# Cell 4 - Inspect Bronze tables
for table in ["customers", "products", "orders"]:
    
    print(f"\n{'='*50}") # =======================
    print(f"{BRONZE_SCHEMA}.{table}") # table name
    print('='*50) # ==============================

    df = spark.table(f"{BRONZE_SCHEMA}.{table}")
    df.printSchema()
    
    print(f"Total rows: {df.count():,}")
    print(f"Null counts:")
    from pyspark.sql.functions import col, sum as spark_sum
    df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()


workspace.retailpulse_bronze.customers
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

Total rows: 100,000
Null counts:
+-----------+----+----+-------+-----------+-------------+
|customer_id|name|city|country|signup_date|_rescued_data|
+-----------+----+----+-------+-----------+-------------+
|          0|   0|   0|      0|          0|       100000|
+-----------+----+----+-------+-----------+-------------+


workspace.retailpulse_bronze.products
root
 |-- product_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)
 |-- stock_qty: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

Total rows: 10,000
Null counts:
+----------+----+--------+-----+---------+-------------+
|product_id|n

#### 3. Clean Customers Table and Write to Silver
This cell performs data cleaning on the `customers` table from the Bronze layer:
- Drops the `_rescued_data` column for containing only null values.
- Casts `customer_id` to integer.
- Converts `signup_date` to a proper date format.
- Removes duplicate customers based on `customer_id`.
The cleaned DataFrame is then ready for writing to the Silver layer.

In [0]:
# Cell 6 - Clean customers → Silver
from pyspark.sql.functions import col, to_date

df_customers = (
    spark.table(f"{BRONZE_SCHEMA}.customers")
    .drop("_rescued_data")
    .withColumn("customer_id", col("customer_id").cast("integer"))
    .withColumn("signup_date", to_date(col("signup_date"), "yyyy-MM-dd"))
    .dropDuplicates(["customer_id"])
)

print(f"Silver customers row count: {df_customers.count():,}")
df_customers.printSchema()
display(df_customers.head(3))

Silver customers row count: 50,000
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)



customer_id,name,city,country,signup_date
12,Linda Harris,Hamburg,DE,2018-11-19
18,Susan Moore,Montreal,CA,2018-10-20
38,Emma Hernandez,Phoenix,US,2023-12-13


#### 4. Clean Products Table and Write to Silver
This cell cleans the `products` table from the Bronze layer:
- Drops the `_rescued_data` column.
- Casts `product_id` and `stock_qty` to integer.
- Casts `price` to decimal with two decimal places.
- Removes duplicate products based on `product_id`.
The cleaned DataFrame is then ready for writing to the Silver layer.

In [0]:
# Cell 8 - Clean products → Silver
from pyspark.sql.functions import col, to_date

df_products = (
    spark.table(f"{BRONZE_SCHEMA}.products")
    .drop("_rescued_data")
    .withColumn("product_id", col("product_id").cast("integer"))
    .withColumn("price", col("price").cast("decimal(10,2)"))
    .withColumn("stock_qty", col("stock_qty").cast("integer"))
    .dropDuplicates(["product_id"])
)

print(f"Silver products row count: {df_products.count():,}")
df_products.printSchema()
display(df_products.head(3))

Silver products row count: 5,000
root
 |-- product_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- stock_qty: integer (nullable = true)



product_id,name,category,price,stock_qty
12,Camping Tent 562,Sports,119.460000000000000000,1924
18,Self-Help Book 534,Books,67.870000000000000000,3050
38,Boots 362,Clothing,196.590000000000000000,584



#### 5. Clean Orders Table and Write to Silver
This cell cleans the `orders` table from the Bronze layer:
- Drops the `_rescued_data` column.
- Casts `order_id`, `customer_id`, `product_id`, and `quantity` to integer.
- Casts `amount` to decimal with two decimal places.
- Converts `order_date` to a proper date format.
- Removes duplicate orders based on `order_id`.
The cleaned DataFrame is then ready for writing to the Silver layer.

In [0]:
# Cell 10 - Clean orders → Silver
from pyspark.sql.functions import col, to_date

df_orders = (
    spark.table(f"{BRONZE_SCHEMA}.orders")
    .drop("_rescued_data")
    .withColumn("order_id", col("order_id").cast("integer"))
    .withColumn("customer_id", col("customer_id").cast("integer"))
    .withColumn("product_id", col("product_id").cast("integer"))
    .withColumn("quantity", col("quantity").cast("integer"))
    .withColumn("amount", col("amount").cast("decimal(10,2)"))
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
    .dropDuplicates(["order_id"])
)

print(f"Silver orders row count: {df_orders.count():,}")
df_orders.printSchema()
display(df_orders.head(3))

Silver orders row count: 500,000
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)



order_id,customer_id,product_id,quantity,amount,order_date,status
12,6897,4496,5,100.800000000000000000,2020-12-30,pending
18,45987,3282,4,506.200000000000000000,2022-02-14,pending
38,12272,3344,7,360.570000000000000000,2021-03-08,pending


   
#### 6. Write Cleaned DataFrames to Silver Delta Tables
This cell defines a helper function to write cleaned DataFrames to the Silver layer as Delta tables:
- The `write_to_silver` function writes a DataFrame to the specified Silver table using overwrite mode.
- The cleaned `customers`, `products`, and `orders` DataFrames are written to their respective Silver tables.
- Confirms successful writes for each table.

In [0]:
# Cell 12 - Write to Silver Delta tables
def write_to_silver(df, table_name):
    target_table = f"{SILVER_SCHEMA}.{table_name}"
    print(f"Writing to {target_table} ...")
    (df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )
    print(f"✅ Done: {target_table}")

write_to_silver(df_customers, "customers")
write_to_silver(df_products, "products")
write_to_silver(df_orders, "orders")

print("\n✅ All Silver tables written!")

Writing to workspace.retailpulse_silver.customers ...
✅ Done: workspace.retailpulse_silver.customers
Writing to workspace.retailpulse_silver.products ...
✅ Done: workspace.retailpulse_silver.products
Writing to workspace.retailpulse_silver.orders ...
✅ Done: workspace.retailpulse_silver.orders

✅ All Silver tables written!


#### 7. Verify Silver Tables
 
This cell verifies the contents of the cleaned Silver tables:
- Iterates through the `customers`, `products`, and `orders` tables in the Silver schema.
- Prints the row count and number of columns for each table.
- Displays the first 3 rows of each table for a quick data check.
This ensures that the Silver tables were written correctly and contain the expected data.

In [0]:
# Cell 14 - Verify Silver tables
for table in ["customers", "products", "orders"]:
    df = spark.table(f"{SILVER_SCHEMA}.{table}")
    print(f"{SILVER_SCHEMA}.{table}: {df.count():,} rows | {len(df.columns)} columns")
    df.show(3)
    print()

workspace.retailpulse_silver.customers: 50,000 rows | 5 columns
+-----------+--------------+--------+-------+-----------+
|customer_id|          name|    city|country|signup_date|
+-----------+--------------+--------+-------+-----------+
|         12|  Linda Harris| Hamburg|     DE| 2018-11-19|
|         18|   Susan Moore|Montreal|     CA| 2018-10-20|
|         38|Emma Hernandez| Phoenix|     US| 2023-12-13|
+-----------+--------------+--------+-------+-----------+
only showing top 3 rows

workspace.retailpulse_silver.products: 5,000 rows | 5 columns
+----------+------------------+--------+------+---------+
|product_id|              name|category| price|stock_qty|
+----------+------------------+--------+------+---------+
|        12|  Camping Tent 562|  Sports|119.46|     1924|
|        18|Self-Help Book 534|   Books| 67.87|     3050|
|        38|         Boots 362|Clothing|196.59|      584|
+----------+------------------+--------+------+---------+
only showing top 3 rows

workspace.re